In [ ]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 5

In [ ]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from metrics import information_coefficient, rank_information_coefficient
from features import add_features

In [ ]:
DATA_DIR="binance_data"
MODEL_DIR = "artifacts"
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_1m.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_rf_model.joblib")
features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(DATA_DIR, f"{SYMBOL}__{TARGET_HORIZON}rf_predictions.csv")

def time_split(df: pd.DataFrame, train_frac: float = 0.8) -> tuple[pd.DataFrame, pd.DataFrame]:
    split_idx = int(len(df) * train_frac)
    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()
    return train_df, test_df

In [ ]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

In [ ]:
target_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col]].copy()
model_df = model_df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

In [ ]:
def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 800, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical(
            "max_features", ["sqrt", "log2", 0.3, 0.5, 0.8, 1.0]
        ),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        "random_state": 42,
        "n_jobs": -1,
    }

    model = RandomForestRegressor(**params)
    model.fit(X_train, y_train)

    pred_valid = model.predict(X_valid)
    rmse = mean_squared_error(y_valid, pred_valid, squared=False)

    return rmse  # minimise RMSE


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("\n[optuna] best trial")
print(f"value: {study.best_value:.6f}")
print("params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
best_params = study.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1

X_train_full = train_df[feature_cols]
y_train_full = train_df[target_col]

final_model = RandomForestRegressor(**best_params)

start = time.time()
print("[training] fitting final random forest...")
final_model.fit(X_train_full, y_train_full)
print(f"[training] done in {time.time() - start:.2f}s")

In [ ]:
train_pred = final_model.predict(X_train_full)
test_pred = final_model.predict(X_test)

In [ ]:
# evaluate
print("[eval] computing metrics...")
train_ic = information_coefficient(y_train_full.values, train_pred)
test_ic = information_coefficient(y_test.values, test_pred)

train_rank_ic = rank_information_coefficient(y_train_full.values, train_pred)
test_rank_ic = rank_information_coefficient(y_test.values, test_pred)

train_rmse = np.sqrt(mean_squared_error(y_train_full, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print("\n===== RESULTS =====")
print(f"Train IC:      {train_ic:.6f}")
print(f"Test IC:       {test_ic:.6f}")
print(f"Train Rank IC: {train_rank_ic:.6f}")
print(f"Test Rank IC:  {test_rank_ic:.6f}")
print(f"Train RMSE:    {train_rmse:.6f}")
print(f"Test RMSE:     {test_rmse:.6f}")

In [ ]:
# feature importance
importances = pd.Series(
    final_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(importances)

In [ ]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")

In [ ]:
# save model
joblib.dump(final_model, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

# save feature importance
importances.to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": TARGET_HORIZON,
    "target_col": target_col,
    "buy_threshold": BUY_THRESHOLD,
    "sell_threshold": SELL_THRESHOLD,
    "rf_params": best_params,
    "n_features": len(feature_cols),
    "feature_cols_path": features_path,
    "model_path": model_path,
    "feature_importance_path": fi_path,
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")